# Notebook 1: Regresión Lineal y Optimización Continua

## ¿Qué es un Problema de Regresión?
El aprendizaje supervisado se divide clásicamente en dos grandes áreas: **Clasificación** (predecir categorías discretas) y **Regresión**.
Un **Problema de Regresión** consiste en construir un modelo matemático que permita predecir un valor numérico continuo basado en una o múltiples características (variables predictoras o independientes).

Ejemplos típicos:
- Predecir el precio de una vivienda (variable continua $y$) en función de su tamaño, ubicación y número de habitaciones ($X$).
- Estimar la cantidad de ventas del próximo mes para un retail, basándose en la inversión en marketing y datos históricos.
- Proyectar la carga térmica de un edificio en función de sus características geométricas y el clima.

El objetivo consiste en deducir la relación y los patrones estructurales subyacentes en los datos (fase de entrenamiento) de tal modo que, al presentarle información inédita al modelo, éste arroje como resultado un valor numérico confiable (fase de inferencia).

## Formulación del Problema
En esta sección abordaremos la regresión lineal, que busca modelar matemáticamente esta relación cuantitativa entre variables predictoras ($X$) y una variable objetivo continua ($y$).

### Ecuación Vectorizada de la Hipótesis
Asumiendo un conjunto de características $x$, el modelo de regresión lineal formula su predicción, la **hipótesis** $h_\theta(x)$, como un producto punto entre el vector de parámetros $\theta$ y el vector de características $x$:

$$ h_\theta(x) = \theta^T x \tag{1} $$

Donde en la **Ecuación (1)**:
- **$h_\theta(x)$**: Representa la **hipótesis** o la predicción. Es el resultado (valor esperado) que calcula el modelo tras ponderar la información.
- **$x$**: Es el **vector de características** (features o atributos) de una observación puntual. Contiene la información de entrada (ej. tamaño, número de piezas). Por convención algebraica, suele incluir un término inicial $x_0 = 1$ para interactuar con el sesgo (bias).
- **$\theta$**: Es el **vector de parámetros** (los "thetas", pesos o *weights*). Estos son los números que el algoritmo de Machine Learning debe "aprender" y ajustar durante el entrenamiento para ponderar correctamente qué nivel de influencia cuantitativa tiene cada valor de $x$ sobre el resultado de $h$.

<div class="alert alert-info">
<b>Recordatorio de Álgebra Lineal:</b><br><br>
En la <b>Ecuación (1)</b>, observarás la expresión $\theta^T x$. Esto denota el <b>producto punto</b> (multiplicación de matrices) entre dos vectores.<br>
Si asumimos que tanto $\theta$ como $x$ son vectores columna de tamaño $(n+1) \times 1$:
$$ \theta = \begin{bmatrix} \theta_0 \\ \theta_1 \\ \vdots \\ \theta_n \end{bmatrix}, \quad x = \begin{bmatrix} x_0 \\ x_1 \\ \vdots \\ x_n \end{bmatrix} $$
Al aplicar la operación <b>transpuesta</b> a $\theta$ (denotado $\theta^T$), matemáticamente "acostamos" el vector columna para transformarlo en un <b>vector fila</b> de tamaño $1 \times (n+1)$. 
Esto es vital porque, por las reglas del álgebra matricial, necesitas emparejar las dimensiones internas. Así, un vector fila interactuando con un vector columna resulta al multiplicar término a término y sumar todo, colapsando el arreglo en un escalar (un número único):
$$ h_\theta(x) = \theta^T x = \begin{bmatrix} \theta_0 & \theta_1 & \dots & \theta_n \end{bmatrix} \begin{bmatrix} x_0 \\ x_1 \\ \vdots \\ x_n \end{bmatrix} = \theta_0 x_0 + \theta_1 x_1 + \dots + \theta_n x_n $$
La notación $\theta^T x$ nos permite evadir escribir grandes sumatorias algebraicas, y permite paralelizar el código usando matrices enormes simultáneamente.

</div>

### Función de Costo: Error Cuadrático Medio (MSE)
Para evaluar la calidad de nuestras estimaciones, necesitamos medir la distancia entre las predicciones $h_\theta(x^{(i)})$ y los valores reales $y^{(i)}$ para cada instancia del dataset. El **Error Cuadrático Medio** (MSE) promedia el cuadrado de estos errores:

$$ J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} \left( h_\theta(x^{(i)}) - y^{(i)} \right)^2 \tag{2} $$

<div class="alert alert-warning">
<b>Profundización Analítica:</b>
<br><br>
Demostración estadística de por qué el MSE es el estimador óptimo bajo la asunción de errores con distribución normal.
<br><br>
Asumamos que el valor real $y^{(i)}$ es generado por la hipótesis más un término de error aleatorio $\epsilon^{(i)} \sim \mathcal{N}(0, \sigma^2)$:
$$ y^{(i)} = \theta^T x^{(i)} + \epsilon^{(i)} \tag{3} $$

La probabilidad (verosimilitud) de observar un $y^{(i)}$ dado $x^{(i)}$ condicionado a $\theta$ es:
$$ p(y^{(i)} | x^{(i)} ; \theta) = \frac{1}{\sqrt{2\pi}\sigma} \exp\left( -\frac{(y^{(i)} - \theta^T x^{(i)})^2}{2\sigma^2} \right) \tag{4} $$

Para todo el dataset, la función de log-verosimilitud $\ell(\theta)$ es:
$$ \ell(\theta) = m \log\left(\frac{1}{\sqrt{2\pi}\sigma}\right) - \frac{1}{2\sigma^2} \sum_{i=1}^{m} (y^{(i)} - \theta^T x^{(i)})^2 \tag{5} $$

Maximizar la probabilidad (MLE) equivale a minimizar el término negativo de la suma de errores al cuadrado, justificando formalmente la función de costo MSE.

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Generación de dataset sintético
np.random.seed(42)
m = 100
X = 2 * np.random.rand(m, 1)
y = 4 + 3 * X + np.random.randn(m, 1)

# 2. Programación de MSE
def calcular_mse(X_b, y, theta):
    m = len(y)
    predicciones = X_b.dot(theta)
    costo = (1 / (2 * m)) * np.sum((predicciones - y) ** 2)
    return costo

X_b = np.c_[np.ones((m, 1)), X]
theta_random = np.random.randn(2, 1)
print(f"MSE Inicial: {calcular_mse(X_b, y, theta_random):.4f}")

## Optimización y Gradiente Descendente
Dado que nuestra meta es minimizar la función de costo $J(\theta)$, necesitamos un algoritmo para encontrar el $\theta$ óptimo a través de la optimización continua.

### ¿Qué es el Descenso del Gradiente?
El **Descenso del Gradiente** (Gradient Descent) es uno de los algoritmos de optimización iterativa más importantes en Machine Learning. Su propósito fundamental es afinar automática y sistemáticamente los parámetros (los pesos $\theta$) para encontrar el punto donde el error de nuestro modelo (la función de costo $J(\theta)$) es el mínimo absoluto.

**La intuición visual:**
Imagina que te han vendado los ojos en la ladera montañosa de un terreno irregular y tu misión es alcanzar rápidamente el fondo del valle (el mínimo de error). Al no poder ver, utilizas tus pies para sentir la inclinación del suelo, determinando hacia qué ángulo la bajada es más pronunciada. Das un paso en esa dirección exacta y luego vuelves a evaluar la pendiente. Repites este proceso interactivamente hasta que sientes que el terreno se aplana, lo que indica que has llegado al fondo.

Traducido al lenguaje matemático, la "pendiente" equivale al derivado de la función, y las iteraciones componen la regla de actualización.

### Regla de Actualización
La regla de actualización del Gradiente Descendente se define iterativamente como:

$$ \theta := \theta - \alpha \nabla_{\theta} J(\theta) \tag{6} $$

Analizando sus componentes:
- **$\theta$**: Es nuestro estado actual; las coordenadas que determinan nuestra posición en la montaña.
- **$\nabla_{\theta} J(\theta)$ (Vector Gradiente)**: Es la "brújula de pendiente". Se calcula tomando las derivadas parciales de la función de costo. Siempre apunta en la dirección de máximo *ascenso*. Como queremos descender, invertimos su dirección restándolo.
- **$\alpha$ (Tasa de Aprendizaje o *Learning Rate*)**: Define el tamaño de cada "paso" que damos en la montaña. Un $\alpha$ muy pequeño resultará en un descenso excesivamente prolongado y lento. Un $\alpha$ gigantesco provocará que los pasos sean tan abruptos que brinquemos sobre el valle, rebotando hacia los picos sin nunca lograr descender del todo.


<div class="alert alert-warning">
<b>Profundización Analítica:</b>
<br><br>
Derivación analítica paso a paso del vector gradiente a partir de la función de costo y análisis de convexidad.
<br><br>
Costo MSE en forma matricial:
$$ J(\theta) = \frac{1}{2m} (X\theta - y)^T(X\theta - y) \tag{7} $$

Derivando el vector gradiente $\nabla_{\theta} J(\theta)$:
$$ \nabla_{\theta} J(\theta) = \frac{1}{m} X^T (X\theta - y) \tag{8} $$

<b>Análisis de la Matriz Hessiana para Convexidad:</b>
Calculamos la matriz Hessiana:
$$ H = \nabla_{\theta}^2 J(\theta) = \frac{1}{m} X^T X \tag{9} $$
Dado cualquier vector $z \neq 0$, $z^T H z = \frac{1}{m} \|Xz\|^2 \geq 0$.
La matriz Hessiana es Positiva Semi-Definida, garantizando que MSE es una función convexa.

</div>

In [ ]:
# Ciclo de Entrenamiento
alpha = 0.1
n_iterations = 100
theta = np.random.randn(2, 1)
cost_history = []

for iter in range(n_iterations):
    error = X_b.dot(theta) - y
    gradiente = (1 / m) * X_b.T.dot(error)
    theta = theta - alpha * gradiente
    cost_history.append(calcular_mse(X_b, y, theta))

# Visualización
plt.plot(range(n_iterations), cost_history, 'r-')
plt.title('Curva de Aprendizaje')
plt.xlabel('Iteraciones')
plt.ylabel('Costo MSE')
plt.show()